# 01 — Thermal Parameters: $c_\alpha$, $d_\alpha$, $V_\alpha$, $\Gamma_\alpha$

**PDF reference**: §2.1, Eq. (2.3)–(2.5)

**Reproduces**: Figure 1 — $c_\alpha(T)$ and $d_\alpha(T)$ for $e$, $\mu$, $\tau$ flavors

## Scope

- Production rate $\Gamma_\alpha = c_\alpha G_F^2 p T^4$
- Effective potential $V_\alpha$ (Eq. 2.4) with energy densities $\rho_{\nu_\alpha}$, $\rho_{\ell_\alpha}$
- Ratio $d_\alpha \equiv -V_\alpha / \Gamma_\alpha$
- Numerical $c_\alpha(T)$ from Ref. [18] data (Ref. [18]: https://laine.itp.unibe.ch/neutrino_rates/)

In [ ]:
using HNL_BBN_Code
using NaturalUnits

using CairoMakie, LaTeXStrings
theme_latexfonts() |> set_theme!

## Step 1: $c_\alpha(T)$ from numerical data

The neutrino production rate is $\Gamma_\alpha = c_\alpha G_F^2\, p\, T^4$ (Eq. 2.3).

$c_\alpha(T)$ is loaded from the numerical lattice-QCD-corrected data of Ref. [13] (hep-ph/0612182, table from Ref. [18]),
evaluated at $q/T = 3$ (representative thermal momentum).

In [ ]:
# Temperature range: 1 MeV to 10 GeV (matching Fig 1)
T_MeV_range = geomspace(1.0, 1e4, 300)  # in MeV
T_list = MeV.(T_MeV_range)

# Compute c_α(T) for each flavor
c_e_vals  = [cα(1, T) for T in T_list]
c_μ_vals  = [cα(2, T) for T in T_list]
c_τ_vals  = [cα(3, T) for T in T_list]

println("c_e(1 MeV) = ", round(c_e_vals[1]; digits=3))
println("c_e(10 GeV) = ", round(c_e_vals[end]; digits=3))
println("c_μ(1 MeV) = ", round(c_μ_vals[1]; digits=3))
println("c_τ(1 MeV) = ", round(c_τ_vals[1]; digits=3))

## Step 2: $V_\alpha(T,p)$ and $d_\alpha(T)$

The effective potential (Eq. 2.4):
$$V_\alpha = -\frac{8\sqrt{2}}{3}\, G_F \left(\frac{\rho_{\nu_\alpha}}{m_Z^2} + \frac{\rho_{\ell_\alpha}}{m_W^2}\right) p$$

The ratio $d_\alpha \equiv -V_\alpha / \Gamma_\alpha$ is momentum-independent (Eq. 2.5).

The right panel of Fig 1 shows $d_\alpha \cdot c_\alpha$, which is also $c_\alpha$-independent:
$$d_\alpha \, c_\alpha = \frac{8\sqrt{2}}{3\, G_F\, T^4} \left(\frac{\rho_{\nu_\alpha}}{m_Z^2} + \frac{\rho_{\ell_\alpha}}{m_W^2}\right)$$

In [ ]:
# Compute d_α(T) for each flavor
d_e_vals  = [dα(1, T) for T in T_list]
d_μ_vals  = [dα(2, T) for T in T_list]
d_τ_vals  = [dα(3, T) for T in T_list]

# d_α × c_α
dc_e_vals = d_e_vals .* c_e_vals
dc_μ_vals = d_μ_vals .* c_μ_vals
dc_τ_vals = d_τ_vals .* c_τ_vals

println("d_e(1 MeV)  = ", round(d_e_vals[1]; digits=2))
println("d_e(10 GeV) = ", round(d_e_vals[end]; digits=2))
println("d_e c_e (high T, asymptotic) ≈ ", round(dc_e_vals[end]; digits=2))

### Verify: high-$T$ limit of $V_\alpha$

At $T \gg m_{\ell_\alpha}$, using $g_\nu = 7/4$ and $g_\ell = 7/2$:

$$V_\alpha \approx -\frac{8\sqrt{2}}{3}\, G_F\, \frac{\pi^2}{30} \left(\frac{7/4}{m_Z^2} + \frac{7/2}{m_W^2}\right) T^4\, p \approx -1.088 \times 10^{-8}\, p\, (T/\mathrm{GeV})^4$$

In [ ]:
# Verify the numerical coefficient at T ≫ all lepton masses
V_coeff = (8sqrt(2)/3) * EUval(GeV, G_F) * (π^2/30) * (7/4 / EUval(GeV, m_Z)^2 + 7/2 / EUval(GeV, m_W)^2)
println("V_α coefficient: ", round(V_coeff; sigdigits=4), " × p × (T/GeV)⁴")
println("Expected:         1.088 × 10⁻⁸")

## Step 3: Figure 1 — $c_\alpha(T)$ and $d_\alpha c_\alpha(T)$

In [ ]:
fig = Figure(size=(900, 400))

# ─── Left panel: c_α(T) ───
ax1 = Axis(fig[1, 1];
    xlabel=L"$T$ [MeV]",
    ylabel=L"$c_\alpha$",
    xscale=log10,
    xticks=[1e0, 1e1, 1e2, 1e3, 1e4],
    yticks=0:2:10,
    limits=(1, 1e4, 0, 10),
)

lines!(ax1, T_MeV_range, c_e_vals; label=L"e", color=:royalblue, linewidth=2)
lines!(ax1, T_MeV_range, c_μ_vals; label=L"\mu", color=:darkorange, linewidth=2)
lines!(ax1, T_MeV_range, c_τ_vals; label=L"\tau", color=:green, linewidth=2)
axislegend(ax1; position=:rb)

# ─── Right panel: d_α c_α(T) ───
ax2 = Axis(fig[1, 2];
    xlabel=L"$T$ [MeV]",
    ylabel=L"$d_\alpha \, c_\alpha$",
    xscale=log10,
    xticks=[1e0, 1e1, 1e2, 1e3, 1e4],
    yticks=0:10:90,
    limits=(1, 1e4, 0, 90),
)

lines!(ax2, T_MeV_range, dc_e_vals; label=L"e", color=:royalblue, linewidth=2)
lines!(ax2, T_MeV_range, dc_μ_vals; label=L"\mu", color=:darkorange, linewidth=2)
lines!(ax2, T_MeV_range, dc_τ_vals; label=L"\tau", color=:green, linewidth=2)
axislegend(ax2; position=:rb)

fig

In [ ]:
# Save the figure
save(joinpath(plot_directory, "fig1_c_alpha_d_alpha.pdf"), fig)
println("Figure saved to plots/fig1_c_alpha_d_alpha.pdf")